# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset described in the Croissant format using the `mlcroissant` library. All dataset entities such as record sets and fields are referenced by their `@id`.

### Dataset Source
The dataset Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Install the `mlcroissant` library if needed
!pip install -U mlcroissant


## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # meta is a DatasetMetadata object

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields for downstream processing.

In [ ]:
# Get record sets and explore their ids, names, and fields via Croissant schema structure

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets in this dataset.")

for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name')}")
    # Fields in the record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields @id list:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id')}")
        else:
            print(f"    - {f}")

## 3. Data Extraction
Extract records from each record set into separate pandas DataFrames for further data analysis.

> **Tip:** All data inputs and manipulations refer to entities by their `@id`.

In [ ]:
# Get the list of record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Build dataframe even if empty
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}")

# Display column names for the first record set, if available
if record_set_ids:
    df0 = dataframes[record_set_ids[0]]
    print(f"\nColumns in record set {record_set_ids[0]}:")
    print(df0.columns.tolist())
    df0.head()
else:
    print('No record sets available in this dataset!')

## 4. Exploratory Data Analysis (EDA)
Apply processing steps such as filtering, normalization, and grouping to numeric fields of a chosen record set, using `@id` references.

> For this example, we work with the first record set (if it exists) and try to identify numeric fields to operate on. Update the `numeric_field_id` and `group_field_id` variables with actual `@id` values as appropriate for your dataset.

In [ ]:
# EDA on the first available record set (if any records and numeric fields exist)

if record_set_ids and not dataframes[record_set_ids[0]].empty:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to auto-detect a numeric field by checking dtypes
    numeric_fields = df.select_dtypes('number').columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # use the @id column name
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() else 0

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        pprint.pprint(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by another field if available
        other_fields = [c for c in df.columns if c != numeric_field_id]
        group_field_id = other_fields[0] if other_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (means):")
            print(grouped_df.head())
    else:
        print('No numeric fields found in this record set for EDA.')
else:
    print('No record sets or no records found for EDA.')

## 5. Visualization
Visualize data: plot distribution of a numeric field or relationships between two columns, referencing fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If there is filtered data and a numeric column, plot its distribution
if 'filtered_df' in locals() and not filtered_df.empty and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If group_field_id exists and has a small number of levels, boxplot
    if 'group_field_id' in locals() and group_field_id and filtered_df[group_field_id].nunique() < 10:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print('No numeric data available for visualization.')

## 6. Conclusion
We have loaded the dataset via its Croissant schema, explored available record sets and their fields (referenced by `@id`), and demonstrated how to filter, normalize, group, and visualize the data using the `mlcroissant` library.

Further steps may involve domain-specific modeling, additional feature engineering, or advanced analytics using the dataset's rich record structure.